In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm

# Paths
AUDIO_DIR    = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\FSC22\Audio Wise V1.0-20220916T202003Z-001\Audio Wise V1.0"
METADATA_CSV = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\FSC22\Metadata-20220916T202011Z-001\Metadata\Metadata V1.0 FSC22.csv"
OUTPUT_DIR   = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\processed"
CSV_OUTPUT   = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\metadata.csv"
SAMPLE_RATE  = 44100
DURATION     = 5.0
TOP_DB       = 30

print("Libraries Imported Successfully!")

Libraries imported successfully!


In [2]:
THREAT_CLASSES = [
    "Fire",          # wildfire / arson
    "Helicopter",    # aerial poaching surveillance
    "VehicleEngine", # illegal vehicle entry into forest
    "Axe",           # illegal logging
    "Chainsaw",      # illegal logging
    "Generator",     # illegal camp equipment
    "Handsaw",       # illegal logging
    "Firework",      # poaching disturbance
    "Gunshot",       # poaching
    "WoodChop",      # illegal logging
    "TreeFalling",   # result of illegal logging
]

# WILDLIFE → normal forest background sounds (16 classes)
# Rain, Thunderstorm, WaterDrops, Wind, Silence,
# Whistling, Speaking, Footsteps, Clapping,
# Insect, Frog, BirdChirping, WingFlaping,
# Lion, WolfHowl, Squirrel

def get_binary_label(class_name):
    """
    Maps FSC22 class name to binary label.
    Returns 'threat' if class is in THREAT_CLASSES, else 'wildlife'
    Uses exact class name matching — no fuzzy matching
    """
    return "threat" if class_name in THREAT_CLASSES else "wildlife"


# Preview the mapping for all classes
meta_preview = pd.read_csv(METADATA_CSV)
print("Class → Binary Label Mapping:")
print("-" * 35)
for cls in sorted(meta_preview['Class Name'].unique()):
    label = get_binary_label(cls)
    print(f"  {cls:20s} → {label}")
print("-" * 35)
print(f"  Total threat classes  : {sum(1 for c in meta_preview['Class Name'].unique() if get_binary_label(c) == 'threat')}")
print(f"  Total wildlife classes: {sum(1 for c in meta_preview['Class Name'].unique() if get_binary_label(c) == 'wildlife')}")

Class → Binary Label Mapping:
-----------------------------------
  Axe                  → threat
  BirdChirping         → wildlife
  Chainsaw             → threat
  Clapping             → wildlife
  Fire                 → threat
  Firework             → threat
  Footsteps            → wildlife
  Frog                 → wildlife
  Generator            → threat
  Gunshot              → threat
  Handsaw              → threat
  Helicopter           → threat
  Insect               → wildlife
  Lion                 → wildlife
  Rain                 → wildlife
  Silence              → wildlife
  Speaking             → wildlife
  Squirrel             → wildlife
  Thunderstorm         → wildlife
  TreeFalling          → threat
  VehicleEngine        → threat
  WaterDrops           → wildlife
  Whistling            → wildlife
  Wind                 → wildlife
  WingFlaping          → wildlife
  WolfHowl             → wildlife
  WoodChop             → threat
-----------------------------------
  

In [3]:
# Run full pipeline to generate records
meta = pd.read_csv(METADATA_CSV)

def preprocess_audio(file_path, sr=SAMPLE_RATE, duration=DURATION):
    try:
        audio, _ = librosa.load(file_path, sr=sr, mono=True)
        audio, _ = librosa.effects.trim(audio, top_db=TOP_DB)
        target_length = int(sr * duration)
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]
        return audio, sr
    except Exception as e:
        return None, None

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

records = []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Building records"):
    filename     = row["Dataset File Name"]
    class_name   = row["Class Name"]
    binary_label = get_binary_label(class_name)
    file_path    = os.path.join(AUDIO_DIR, filename)
    out_dir      = os.path.join(OUTPUT_DIR, binary_label)
    out_name     = f"{class_name}_{filename}"
    out_path     = os.path.join(out_dir, out_name)

    records.append({
        "filename"      : out_name,
        "original_class": class_name,
        "label"         : binary_label,
        "label_encoded" : 1 if binary_label == "threat" else 0,
        "sample_rate"   : SAMPLE_RATE,
        "duration_sec"  : DURATION,
        "processed_path": out_path
    })

# Save to CSV
df = pd.DataFrame(records)
df.to_csv(CSV_OUTPUT, index=False)

print(f"metadata.csv saved to: {CSV_OUTPUT}")
print(f"Total records        : {len(df)}")
print(f"\nColumn names for Team 2:")
print(df.columns.tolist())
print(f"\nSample rows:")
print(df[['filename', 'original_class', 'label', 'label_encoded']].head())

Building records: 100%|██████████| 2025/2025 [00:00<00:00, 21706.32it/s]

metadata.csv saved to: C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\metadata.csv
Total records        : 2025

Column names for Team 2:
['filename', 'original_class', 'label', 'label_encoded', 'sample_rate', 'duration_sec', 'processed_path']

Sample rows:
           filename original_class   label  label_encoded
0  Fire_1_10101.wav           Fire  threat              1
1  Fire_1_10102.wav           Fire  threat              1
2  Fire_1_10103.wav           Fire  threat              1
3  Fire_1_10104.wav           Fire  threat              1
4  Fire_1_10105.wav           Fire  threat              1


In [4]:
print("\n" + "="*50)
print("   PREPROCESSING COMPLETE — FSC22")
print("="*50)
print(f"  Total processed  : {len(df)}")
print(f"  Wildlife (0)     : {(df['label'] == 'wildlife').sum()}")
print(f"  Threat   (1)     : {(df['label'] == 'threat').sum()}")
print(f"  Output folder    : {OUTPUT_DIR}")
print(f"  CSV for Team 2   : {CSV_OUTPUT}")
print("="*50)

print("\n THREAT class breakdown (11 classes × 75 files):")
print(df[df['label'] == 'threat']['original_class'].value_counts().to_string())

print("\n WILDLIFE class breakdown (16 classes × 75 files):")
print(df[df['label'] == 'wildlife']['original_class'].value_counts().to_string())

print("\n" + "="*50)
print("  Handoff to Team 2 (Feature Extraction) — READY")
print("  Team 2 reads: data/metadata.csv")
print("  Column to use: processed_path")
print("="*50)


   PREPROCESSING COMPLETE — FSC22
  Total processed  : 2025
  Wildlife (0)     : 1200
  Threat   (1)     : 825
  Output folder    : C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\processed
  CSV for Team 2   : C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\metadata.csv

 THREAT class breakdown (11 classes × 75 files):
original_class
Fire             75
TreeFalling      75
Helicopter       75
VehicleEngine    75
Axe              75
Chainsaw         75
Generator        75
Handsaw          75
Firework         75
Gunshot          75
WoodChop         75

 WILDLIFE class breakdown (16 classes × 75 files):
original_class
Rain            75
Thunderstorm    75
WaterDrops      75
Wind            75
Silence         75
Whistling       75
Speaking        75
Footsteps       75
Clapping        75
Insect          75
Frog            75
BirdChirping    75
WingFlaping     75
Lion            75
WolfHowl        75
Squirrel        75

  Handoff to Team 2 (Feature Extraction) — READY
  Team 2 